In [1]:
import pandas as pd
import numpy as np

from datetime import datetime, timedelta

from sklearn.preprocessing import MinMaxScaler

In [2]:
hotspot_df = pd.read_csv("gridlock_final_dataset.csv")

cluster_summary = pd.read_csv("cluster_summary.csv")

In [33]:
cols = [
    "density",
    "avg_vehicle_weight",
    "avg_severity",
    "avg_repeat",
    "avg_criticality"
]

historical_scaler = MinMaxScaler()

historical_scaler.fit(
    cluster_summary[cols]
)

,"feature_range feature_range: tuple (min, max), default=(0, 1)Desired range of transformed data.","(0, ...)"
,"copy copy: bool, default=TrueSet to False to perform inplace row normalization and avoid acopy (if the input is already a numpy array).",True
,"clip clip: bool, default=FalseSet to True to clip transformed values of held-out data toprovided `feature_range`.Since this parameter will clip values, `inverse_transform` may notbe able to restore the original data... note:: Setting `clip=True` does not prevent feature drift (a distribution shift between training and test data). The transformed values are clipped to the `feature_range`, which helps avoid unintended behavior in models sensitive to out-of-range inputs (e.g. linear models). Use with care, as clipping can distort the distribution of test data... versionadded:: 0.24",False


In [3]:
SCENARIOS = {
    "normal": 1.0,
    "festival": 1.8,
    "rain": 1.3,
    "enforcement": 0.6
}

In [17]:
INTERVENTIONS = {
    "none": 1.00,

    "extra_enforcement": 0.75,

    "tow_truck_drive": 0.60,

    "temporary_parking_zone": 0.55,

    "traffic_diversion": 0.70
}

In [4]:
VEHICLE_DIST = {
    "SCOOTER": 0.35,
    "MOTOR CYCLE": 0.25,
    "CAR": 0.25,
    "PASSENGER AUTO": 0.10,
    "GOODS AUTO": 0.05
}

In [5]:
vehicle_weights = {
    "SCOOTER": 1,
    "MOTOR CYCLE": 1,
    "MOPED": 1,

    "CAR": 2,
    "JEEP": 2,
    "PASSENGER AUTO": 2,

    "MAXI-CAB": 3,
    "VAN": 3,

    "GOODS AUTO": 4,

    "LGV": 5,
    "TEMPO": 5,

    "PRIVATE BUS": 8,
    "BUS (BMTC/KSRTC)": 8,
    "SCHOOL VEHICLE": 8,
    "TOURIST BUS": 8,
    "FACTORY BUS": 8,

    "MINI LORRY": 7,

    "LORRY/GOODS VEHICLE": 10,
    "HGV": 10,
    "TANKER": 10,

    "TRACTOR": 6
}

In [6]:
severity_map = {
    "PARKING IN A MAIN ROAD": 5,
    "DOUBLE PARKING": 5,

    "PARKING ON FOOTPATH": 4,
    "PARKING NEAR ROAD CROSSING": 4,
    "PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS": 4,
    "PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC": 4,

    "NO PARKING": 3,

    "WRONG PARKING": 2
}

In [44]:
def criticality(location):

    if pd.isna(location):
        return 1.0

    location = str(location).lower()

    if "metro" in location:
        return 1.5

    if "market" in location:
        return 1.5

    if "hospital" in location:
        return 1.4

    if "bus stop" in location:
        return 1.4

    if "circle" in location:
        return 1.2

    if "junction" in location:
        return 1.2

    return 1.0

In [8]:
def generate_vehicle_number():

    return (
        "KA"
        + str(np.random.randint(1, 99)).zfill(2)
        + "".join(np.random.choice(list("ABCDEFGHIJKLMNOPQRSTUVWXYZ"), 2))
        + str(np.random.randint(1000, 9999))
    )

In [9]:
VIOLATION_LIBRARY = [
    "NO PARKING",
    "PARKING IN A MAIN ROAD",
    "DOUBLE PARKING",
    "PARKING ON FOOTPATH",
    "WRONG PARKING"
]

In [10]:
def build_violation_mix(dominant):

    mix = {
        dominant: 0.70
    }

    others = [
        v
        for v in VIOLATION_LIBRARY
        if v != dominant
    ]

    chosen = np.random.choice(
        others,
        size=2,
        replace=False
    )

    mix[chosen[0]] = 0.20
    mix[chosen[1]] = 0.10

    return mix

In [45]:
def simulate_hotspot(
    hotspot_name,
    days,
    scenario,
    intervention,
    hotspot_df
):

    hotspot = hotspot_df[
        hotspot_df["hotspot_name"] == hotspot_name
    ].iloc[0]

    center_lat = hotspot["center_lat"]
    center_lon = hotspot["center_lon"]

    violations = hotspot["violations"]

    peak_hour = int(hotspot["peak_hour"])

    dominant_violation = hotspot["dominant_violation"]

    violation_mix = build_violation_mix(
        dominant_violation
    )

    daily_avg = violations / 150

    daily_avg *= SCENARIOS[scenario]
    daily_avg *= INTERVENTIONS[intervention]

    rows = []

    vehicle_pool = []

    start_date = datetime.now()

    for day in range(days):

        date = start_date + timedelta(days=day)

        daily_count = np.random.poisson(
            daily_avg
        )

        for _ in range(daily_count):

            hour = int(
                np.random.normal(
                    peak_hour,
                    1
                )
            )

            hour = max(
                0,
                min(23, hour)
            )

            minute = np.random.randint(
                0,
                60
            )

            timestamp = date.replace(
                hour=hour,
                minute=minute,
                second=0,
                microsecond=0
            )

            lat = (
                center_lat
                + np.random.normal(
                    0,
                    0.0005
                )
            )

            lon = (
                center_lon
                + np.random.normal(
                    0,
                    0.0005
                )
            )

            violation_type = np.random.choice(
                list(violation_mix.keys()),
                p=list(violation_mix.values())
            )

            vehicle_type = np.random.choice(
                list(VEHICLE_DIST.keys()),
                p=list(VEHICLE_DIST.values())
            )

            if (
                len(vehicle_pool) > 0
                and np.random.rand() < 0.20
            ):
                vehicle_number = np.random.choice(
                    vehicle_pool
                )
            else:
                vehicle_number = generate_vehicle_number()
                vehicle_pool.append(
                    vehicle_number
                )

            rows.append({
                "timestamp": timestamp,
                "latitude": lat,
                "longitude": lon,
                "violation_type": violation_type,
                "vehicle_type": vehicle_type,
                "vehicle_number": vehicle_number,
                "hotspot_name": hotspot_name
            })

    return pd.DataFrame(rows)

In [75]:
def recommend_resources(projected_pcri):

    if projected_pcri >= 90:

        return {
            "risk_tier": "CRITICAL",
            "officers": 6,
            "tow_trucks": 3,
            "patrol_interval": "15 mins"
        }

    elif projected_pcri >= 70:

        return {
            "risk_tier": "HIGH",
            "officers": 4,
            "tow_trucks": 2,
            "patrol_interval": "30 mins"
        }

    elif projected_pcri >= 50:

        return {
            "risk_tier": "MEDIUM",
            "officers": 2,
            "tow_trucks": 1,
            "patrol_interval": "1 hour"
        }

    else:

        return {
            "risk_tier": "LOW",
            "officers": 1,
            "tow_trucks": 0,
            "patrol_interval": "4 hours"
        }

In [47]:

def compute_simulated_pcri(
    df_sim,
    historical_scaler
):

    # -------------------------
    # Vehicle Weight
    # -------------------------

    df_sim["vehicle_weight"] = (
        df_sim["vehicle_type"]
        .map(vehicle_weights)
        .fillna(2)
    )

    # -------------------------
    # Severity
    # -------------------------

    df_sim["severity_score"] = (
        df_sim["violation_type"]
        .map(severity_map)
        .fillna(1)
    )

    # -------------------------
    # Repeat Offenders
    # -------------------------

    repeat_counts = (
        df_sim.groupby("vehicle_number")
        .size()
        .rename("vehicle_repeat_count")
    )

    df_sim = df_sim.merge(
        repeat_counts,
        left_on="vehicle_number",
        right_index=True,
        how="left"
    )

    # -------------------------
    # Road Criticality
    # -------------------------

    df_sim["road_criticality"] = (
        df_sim["hotspot_name"]
        .apply(criticality)
    )

    # -------------------------
    # Compactness
    # -------------------------

    lat_std = df_sim["latitude"].std()
    lon_std = df_sim["longitude"].std()

    compactness = np.sqrt(
        lat_std**2 +
        lon_std**2
    )

    # avoid divide-by-zero
    compactness = max(compactness, 1e-6)

    # -------------------------
    # IMPORTANT UPDATE
    # Convert simulated violations
    # back to equivalent 150-day horizon
    # -------------------------

    density = (
    len(df_sim)
    / compactness
)

    # -------------------------
    # Simulated Feature Row
    # -------------------------

    sim_row = {
        "density": density,

        "avg_vehicle_weight":
            df_sim["vehicle_weight"].mean(),

        "avg_severity":
            df_sim["severity_score"].mean(),

        "avg_repeat":
            df_sim["vehicle_repeat_count"].mean(),

        "avg_criticality":
            df_sim["road_criticality"].mean()
    }

    cols = [
        "density",
        "avg_vehicle_weight",
        "avg_severity",
        "avg_repeat",
        "avg_criticality"
    ]

    # -------------------------
    # Append To Historical Data
    # -------------------------

    sim_norm = historical_scaler.transform(
    pd.DataFrame([sim_row])
)[0]

    density_norm = sim_norm[0]
    vehicle_weight_norm = sim_norm[1]
    severity_norm = sim_norm[2]
    repeat_norm = sim_norm[3]
    criticality_norm = sim_norm[4]

    # -------------------------
    # ORIGINAL PCRI FORMULA
    # -------------------------

    pcri = (
          0.35 * density_norm
        + 0.20 * severity_norm
        + 0.15 * vehicle_weight_norm
        + 0.15 * repeat_norm
        + 0.15 * criticality_norm
    ) * 100

    return {
        "PCRI": round(float(pcri), 2),

        "simulated_violations":len(df_sim),

        "density":
            round(float(density), 2),

        "avg_vehicle_weight":
            round(float(sim_row["avg_vehicle_weight"]), 2),

        "avg_severity":
            round(float(sim_row["avg_severity"]), 2),

        "avg_repeat":
            round(float(sim_row["avg_repeat"]), 2),

        "avg_criticality":
            round(float(sim_row["avg_criticality"]), 2),

        "density_norm":
            round(float(density_norm), 4),

        "vehicle_weight_norm":
            round(float(vehicle_weight_norm), 4),

        "severity_norm":
            round(float(severity_norm), 4),

        "repeat_norm":
            round(float(repeat_norm), 4),

        "criticality_norm":
            round(float(criticality_norm), 4)
    }

In [48]:
def compare_intervention(
    hotspot_name,
    days,
    scenario,
    intervention,
    hotspot_df,
    cluster_summary
):
    baseline_df = simulate_hotspot(
    hotspot_name=hotspot_name,
    days=days,
    scenario=scenario,
    intervention="none",
    hotspot_df=hotspot_df
)

    baseline_result = compute_simulated_pcri(
    baseline_df,
    historical_scaler
)

    intervention_df = simulate_hotspot(
    hotspot_name=hotspot_name,
    days=days,
    scenario=scenario,
    intervention=intervention,
    hotspot_df=hotspot_df
)
    
    intervention_result = compute_simulated_pcri(
    intervention_df,
    historical_scaler
)

    baseline_pcri = baseline_result["PCRI"]

    new_pcri = intervention_result["PCRI"]

    improvement_pct = (
    (baseline_pcri - new_pcri)
    / baseline_pcri
) * 100

    resources = recommend_resources(
    new_pcri
)

    return {
    "hotspot": hotspot_name,

    "scenario": scenario,

    "intervention": intervention,

    "baseline_pcri": round(
        baseline_pcri,
        2
    ),

    "new_pcri": round(
        new_pcri,
        2
    ),

    "improvement_pct": round(
        improvement_pct,
        2
    ),

    "baseline_violations":
    baseline_result["simulated_violations"],
    
    "new_violations":
    intervention_result["simulated_violations"],

    "resource_plan":
        resources
}

In [50]:
result = compare_intervention(
    hotspot_name=hotspot_df.iloc[0]["hotspot_name"],
    days=7,
    scenario="festival",
    intervention="temporary_parking_zone",
    hotspot_df=hotspot_df,
    cluster_summary=cluster_summary
)

print(result)

{'hotspot': 'New Horizon College Road, New Horizon College of Engineering, Kadubisanahalli, Bengaluru, Karnataka. Pin-560103 (India)', 'scenario': 'festival', 'intervention': 'temporary_parking_zone', 'baseline_pcri': 9.26, 'new_pcri': 7.62, 'improvement_pct': 17.71, 'baseline_violations': 535, 'new_violations': 320, 'resource_plan': {'risk_tier': 'LOW', 'officers': 1, 'tow_trucks': 0, 'patrol_interval': '4 hours'}}


In [16]:
df_sim = simulate_hotspot(
    hotspot_name=hotspot_df.iloc[0]["hotspot_name"],
    days=7,
    scenario="festival",
    hotspot_df=hotspot_df
)

result = compute_simulated_pcri(
    df_sim,
    cluster_summary,
    simulation_days=7
)

print(result)

{'PCRI': 41.07, 'equivalent_150_day_violations': 11850.0, 'density': 16615126.53, 'avg_vehicle_weight': 1.5, 'avg_severity': 3.44, 'avg_repeat': 1.56, 'avg_criticality': 1.0, 'density_norm': 1.0, 'vehicle_weight_norm': 0.1257, 'severity_norm': 0.137, 'repeat_norm': 0.0967, 'criticality_norm': 0.0}


In [22]:
cluster_summary["density"].describe()

count    1.290000e+02
mean     3.009707e+05
std      6.909842e+05
min      2.666200e+04
25%      5.626529e+04
50%      9.734067e+04
75%      2.434892e+05
max      6.279610e+06
Name: density, dtype: float64

In [23]:
cluster_summary["avg_severity"].describe()

count    129.000000
mean       3.742743
std        1.660581
min        2.016529
25%        2.671053
50%        3.160000
75%        4.360885
max       12.391304
Name: avg_severity, dtype: float64

In [24]:
cluster_summary["avg_repeat"].describe()

count    129.000000
mean       1.510170
std        0.731397
min        1.016129
25%        1.134615
50%        1.257669
75%        1.559772
max        6.648000
Name: avg_repeat, dtype: float64

In [25]:
cluster_summary["avg_vehicle_weight"].describe()

count    129.000000
mean       2.070403
std        0.741544
min        1.000000
25%        1.525253
50%        1.951613
75%        2.376176
max        4.942308
Name: avg_vehicle_weight, dtype: float64

In [27]:
hotspot = hotspot_df.iloc[0]["hotspot_name"]

for scenario in ["normal", "festival", "enforcement"]:

    df_sim = simulate_hotspot(
        hotspot_name=hotspot,
        days=7,
        scenario=scenario,
        intervention="none",
        hotspot_df=hotspot_df
    )

    print(
        scenario,
        "->",
        len(df_sim)
    )

normal -> 323
festival -> 564
enforcement -> 178


In [28]:
for scenario in ["normal", "festival", "enforcement"]:

    df_sim = simulate_hotspot(
        hotspot_name=hotspot,
        days=7,
        scenario=scenario,
        intervention="none",
        hotspot_df=hotspot_df
    )

    lat_std = df_sim["latitude"].std()
    lon_std = df_sim["longitude"].std()

    compactness = np.sqrt(
        lat_std**2 + lon_std**2
    )

    density = len(df_sim) / max(compactness, 1e-6)

    print(
        scenario,
        "violations:",
        len(df_sim),
        "density:",
        round(density, 2)
    )

normal violations: 306 density: 455242.79
festival violations: 584 density: 828334.04
enforcement violations: 213 density: 312966.51


In [29]:
print(cluster_summary["density"].describe())

count    1.290000e+02
mean     3.009707e+05
std      6.909842e+05
min      2.666200e+04
25%      5.626529e+04
50%      9.734067e+04
75%      2.434892e+05
max      6.279610e+06
Name: density, dtype: float64


In [30]:
hotspot = hotspot_df.iloc[0]["hotspot_name"]

df_sim = simulate_hotspot(
    hotspot_name=hotspot,
    days=7,
    scenario="normal",
    intervention="none",
    hotspot_df=hotspot_df
)

lat_std = df_sim["latitude"].std()
lon_std = df_sim["longitude"].std()

compactness = np.sqrt(
    lat_std**2 + lon_std**2
)

density = len(df_sim) / max(compactness, 1e-6)

print("Sim Density:", density)

Sim Density: 449049.0935260627


In [51]:
def get_current_pcri(
    hotspot_name,
    hotspot_df
):

    row = hotspot_df[
        hotspot_df["hotspot_name"] == hotspot_name
    ].iloc[0]

    return float(row["PCRI"])

In [74]:
def project_future_pcri(
    current_pcri,
    historical_violations,
    simulated_violations,
    days
):

    expected_baseline = (
        historical_violations / 150
    ) * days

    factor = (
        simulated_violations /
        expected_baseline
    )

    factor = max(
        0.5,
        min(
            factor,
            2.0
        )
    )

    projected_pcri = (
        current_pcri *
        factor
    )

    return round(
        projected_pcri,
        2
    )

In [76]:
def impact_label(change_pct):

    if change_pct >= 50:
        return "SEVERE"

    elif change_pct >= 25:
        return "HIGH"

    elif change_pct >= 10:
        return "MODERATE"

    else:
        return "LOW"

In [77]:
def simulation_confidence(
    simulated_violations
):

    confidence = min(
        95,
        50 + (
            simulated_violations / 10
        )
    )

    return round(
        confidence,
        1
    )

In [78]:
def scenario_dashboard(
    hotspot_name,
    days,
    hotspot_df,
    historical_scaler
):

    hotspot = hotspot_df[
        hotspot_df["hotspot_name"] == hotspot_name
    ].iloc[0]

    current_pcri = float(
        hotspot["PCRI"]
    )

    historical_violations = int(
        hotspot["violations"]
    )

    scenarios = [

        ("normal", "none"),

        ("rain", "none"),

        ("festival", "none"),

        ("festival", "extra_enforcement"),

        ("festival", "temporary_parking_zone"),

        ("festival", "traffic_diversion")

    ]

    results = []

    for scenario, intervention in scenarios:

        df_sim = simulate_hotspot(
            hotspot_name=hotspot_name,
            days=days,
            scenario=scenario,
            intervention=intervention,
            hotspot_df=hotspot_df
        )

        sim_result = compute_simulated_pcri(
            df_sim,
            historical_scaler
        )

        projected_pcri = project_future_pcri(
            current_pcri=current_pcri,
            historical_violations=historical_violations,
            simulated_violations=
                sim_result[
                    "simulated_violations"
                ],
            days=days
        )

        change_pct = (
            (
                projected_pcri -
                current_pcri
            )
            /
            current_pcri
        ) * 100

        results.append({

            "Scenario":
                scenario,

            "Intervention":
                intervention,

            "Projected_PCRI":
                round(
                    projected_pcri,
                    2
                ),

            "Change_%":
                round(
                    change_pct,
                    2
                ),

            "Impact":
                impact_label(
                    change_pct
                ),

            "Confidence":
                simulation_confidence(
                    sim_result[
                        "simulated_violations"
                    ]
                ),

            "Violations":
                sim_result[
                    "simulated_violations"
                ]
        })

    dashboard = pd.DataFrame(
        results
    )

    dashboard = dashboard.sort_values(
        by="Projected_PCRI",
        ascending=False
    )

    dashboard.reset_index(
        drop=True,
        inplace=True
    )

    return dashboard

In [82]:
def best_intervention(
    dashboard
):

    interventions = dashboard[
        dashboard["Intervention"] != "none"
    ].copy()

    if len(interventions) == 0:
        return None

    best_idx = interventions[
        "Projected_PCRI"
    ].idxmin()

    best = interventions.loc[
        best_idx
    ]

    baseline = dashboard[
    (dashboard["Scenario"] == "festival") &
    (dashboard["Intervention"] == "none")
].iloc[0]

    violations_prevented = (
    baseline["Violations"]
    - best["Violations"]
)

    return {

    "best_strategy":
        str(best["Intervention"]),

    "projected_pcri":
        float(best["Projected_PCRI"]),

    "impact":
        str(best["Impact"]),

    "confidence":
        float(best["Confidence"]),

    "violations":
        int(best["Violations"]),

    "violations_prevented":int(violations_prevented)
}

In [83]:
dashboard = scenario_dashboard(
    hotspot_name=hotspot_df.iloc[0]["hotspot_name"],
    days=7,
    hotspot_df=hotspot_df,
    historical_scaler=historical_scaler
)

print(dashboard)

print("\nBest Intervention\n")

print(
    best_intervention(
        dashboard
    )
)

   Scenario            Intervention  Projected_PCRI  Change_%  Impact  \
0  festival                    none          104.63     85.10  SEVERE   
1  festival       extra_enforcement           79.51     40.66    HIGH   
2      rain                    none           75.56     33.68    HIGH   
3  festival       traffic_diversion           74.84     32.40    HIGH   
4  festival  temporary_parking_zone           55.99     -0.95     LOW   
5    normal                    none           52.76     -6.66     LOW   

   Confidence  Violations  
0        95.0         583  
1        94.3         443  
2        92.1         421  
3        91.7         417  
4        81.2         312  
5        79.4         294  

Best Intervention

{'best_strategy': 'temporary_parking_zone', 'projected_pcri': 55.99, 'impact': 'LOW', 'confidence': 81.2, 'violations': 312, 'violations_prevented': 271}
